# 🔬 **Cross-Organ TME Classification — Merged Architecture Framework**
### **Alzoubi et al. (Sci Rep 2026) Backbones + Novel TME-Structure-Aware Hierarchical Head**

This notebook merges the exact backbones evaluated in **Alzoubi et al. (2026)** (*Swin Transformer*, *ConvNeXtV2*, *UNI2-h*) with your **Hierarchy-Aware Head (`HierarchyAwareHead`)** for 80/20 train/val split cross-organ transfer learning on Google Colab.

**Merged Architecture & Experimental Capabilities:**
1. **Paper Backbones (via `timm`):**
   - `swin`: **Swin-B (Swin Transformer Base)** — Top performer in Alzoubi et al.
   - `convnextv2`: **ConvNeXtV2-Tiny** — Modern CNN with GRN & FCMAE pretraining.
   - `uni`: **UNI / UNI2-h** — Pathology foundation model (MahmoodLab).
   - `v2_s` / `vit`: EfficientNet-V2-S and ViT-B/16 standard options.
2. **Novel Classification Head (`HierarchyAwareHead`):**
   - Level-1 Coarse TME Groups: Epithelial (NOR, TUM), Stromal (STR, MUS, ADI), Other (LYM, MUC, DEB).
   - Toggle `USE_HIERARCHY_HEAD = True` vs `False` for direct A/B testing against standard flat linear heads.
3. **80/20 Stratified / Slide Split:**
   - 80% train and 20% validation split (slide-grouped to prevent patch leakage across slides).
4. **Few-Shot / Label Efficiency Evaluation:**
   - Set `FEW_SHOT_FRACTION` (`0.05`, `0.10`, `0.20`, `1.00`) to test performance at 5%, 10%, 20%, or 100% target labels.
5. **Targeted Confusion & Calibration Analysis:**
   - Tracks `ADI ↔ MUC`, `MUS ↔ STR`, `NOR ↔ TUM` errors and calculates Expected Calibration Error (ECE).

In [1]:
# ── Step 1: Install Dependencies & GPU Check ─────────────────────────────────
import subprocess
import sys

# Install required packages quietly
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-image', 'kagglehub', 'timm', 'seaborn', '-q'], check=True)

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Active: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    print("⚠️ No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU in Colab.")

✅ GPU Active: Tesla T4 (15.64 GB VRAM)


In [2]:
# ── Step 2: Imports & Global Configuration ─────────────────────────────────────
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import timm
from skimage.color import rgb2hed, hed2rgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ── Random Seed ────────────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

# ── Model Architecture Settings ─────────────────────────────────────────────
# Backbone Options:
#   'swin'        : Swin Transformer Base
#   'convnextv2'  : ConvNeXtV2-Tiny (Modern CNN with GRN)
#   'v2_s'        : EfficientNet-V2-S
#   'effb0'       : EfficientNet-B0 (Lou et al. 2025 baseline backbone)
#   'vit'         : ViT-B/16 (Lou et al. 2025 baseline backbone)
#   'uni'         : UNI Pathology Foundation Model (requires HF login + access approval)
MODEL_TYPE          = 'swin'    # Default: Swin Transformer
USE_HIERARCHY_HEAD  = True      # True = HierarchyAwareHead, False = Flat Linear Head
FREEZE_ENCODER      = False     # False = Full end-to-end fine-tuning
FEW_SHOT_FRACTION   = 1.0       # 1.0 = 100% train set, 0.05 = 5%, 0.10 = 10%, 0.20 = 20%
# Checkpoint Selection Metric: 'macro_f1' (recommended), 'acc', 'primary_ce', 'total_loss'
CHECKPOINT_METRIC   = 'macro_f1'

# ── Baseline Replication Mode ────────────────────────────────────────────────
# When True, overrides the settings above to exactly match Lou et al. (2025)'s
# reported training recipe for their ViT / EfficientNet-B0 baselines on this
# same dataset: flat head (no hierarchy), full fine-tuning, single Adam
# optimizer (not AdamW) at lr=1e-3, weight_decay=1e-4, patience=5. Use this to
# produce a like-for-like baseline number before comparing your hierarchy head
# against it — do NOT skip straight to the hierarchy run and assume this
# number; run it and record whatever it actually measures.
BASELINE_REPLICATION_MODE = False   # Set True + MODEL_TYPE in {'vit','effb0'} to replicate paper baselines

if BASELINE_REPLICATION_MODE:
    if MODEL_TYPE not in ['vit', 'effb0']:
        print(f"⚠️ BASELINE_REPLICATION_MODE is on but MODEL_TYPE='{MODEL_TYPE}' is not one of "
              f"Lou et al.'s tested backbones ('vit', 'effb0'). Proceeding anyway, but this will "
              f"not be a direct replication of their reported numbers.")
    USE_HIERARCHY_HEAD = False
    FREEZE_ENCODER      = False
    PATIENCE            = 5
else:
    PATIENCE            = 4

IMG_SIZE            = (224, 224)
BATCH_SIZE          = 32 if MODEL_TYPE in ['swin', 'vit'] else 64
HEAD_LR             = 1e-3
WEIGHT_DECAY        = 1e-4
EPOCHS               = 15
WARMUP_EPOCHS        = 1

# Per-architecture encoder LR (used only when FREEZE_ENCODER=False and NOT in
# baseline replication mode, which instead uses a single lr=1e-3 for everything
# matching Lou et al.'s exact recipe).
ENCODER_LR_CONFIG = {
    'swin':       3e-5,
    'vit':        3e-5,
    'uni':        3e-5,
    'convnextv2': 1e-4,
    'v2_s':       1e-4,
    'effb0':      1e-4,
}
ENCODER_LR = ENCODER_LR_CONFIG.get(MODEL_TYPE, 1e-5)

# Loss Weights
PRIMARY_LOSS_WEIGHT = 1.0   # direct CE on combined_logits (the thing you actually evaluate)
COARSE_LOSS_WEIGHT  = 1.0
FINE_LOSS_WEIGHT    = 1.0
HIER_LOSS_WEIGHT    = 0.5
USE_STAIN_AUGMENTATION = True

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device             : {DEVICE}")
print(f"Backbone           : {MODEL_TYPE.upper()}")
print(f"Hierarchy Head     : {USE_HIERARCHY_HEAD}")
print(f"Freeze Encoder     : {FREEZE_ENCODER}")
print(f"Encoder LR (if unfrozen): {ENCODER_LR}")
print(f"Few-Shot Fraction  : {FEW_SHOT_FRACTION * 100:.0f}%")
print(f"Checkpoint Metric  : {CHECKPOINT_METRIC}")
print(f"Baseline Replication Mode: {BASELINE_REPLICATION_MODE}")
if BASELINE_REPLICATION_MODE:
    print(f"  -> Adam optimizer, lr=1e-3 (single group), weight_decay=1e-4, patience=5, flat head")


Device             : cuda
Merged Backbone    : SWIN (Alzoubi et al. 2026 Paper Backbone)
Hierarchy Head     : True
Freeze Encoder     : False
Encoder LR (if unfrozen): 3e-05
Few-Shot Fraction  : 100%
Checkpoint Metric  : macro_f1


In [3]:
# ── Step 3: Stain Robustness Pipeline & Image Transforms ───────────────────────
class HEDJitter:
    """Perturbs H&E stain channels independently (Tellez et al., 2018)."""
    def __init__(self, sigma=0.05, bias=0.02):
        self.sigma = sigma
        self.bias  = bias

    def __call__(self, img):
        img_np = np.asarray(img).astype(np.float32) / 255.0
        hed    = rgb2hed(img_np)
        alpha  = 1 + np.random.uniform(-self.sigma, self.sigma, size=3)
        beta   = np.random.uniform(-self.bias,  self.bias,  size=3)
        hed    = hed * alpha + beta
        rgb    = np.clip(hed2rgb(hed), 0, 1)
        return Image.fromarray((rgb * 255).astype(np.uint8))

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_train_transforms():
    steps = [transforms.Resize(IMG_SIZE)]
    if USE_STAIN_AUGMENTATION:
        steps.append(HEDJitter())
    steps += [
        transforms.RandomAffine(degrees=20, translate=(0.1, 0.1), scale=(0.8, 1.2)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=_MEAN, std=_STD),
    ]
    return transforms.Compose(steps)

def get_val_transforms():
    return transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=_MEAN, std=_STD),
    ])

print("✅ `get_train_transforms()` & `get_val_transforms()` defined successfully.")

✅ `get_train_transforms()` & `get_val_transforms()` defined successfully.


In [5]:
# ── Step 4: Dataset Location & Kagglehub Download ───────────────────────────────
import kagglehub

print("📥 Checking dataset path / downloading via kagglehub...")
LOCAL_PATHS = [
    '/content/all_image',
    '/content/gastric_cancer/all_image',
    '/content/drive/MyDrive/gastric-cancer/all_image',
    '/content/drive/MyDrive/all_image'
]

DATA_DIR = None
for p in LOCAL_PATHS:
    if os.path.isdir(p):
        DATA_DIR = p
        print(f"✅ Found dataset locally at: {DATA_DIR}")
        break

if DATA_DIR is None:
    try:
        dataset_raw_path = kagglehub.dataset_download("azmainofficial/gastric-cancer")
    except Exception as e:
        print(f"\n⚠️ Kaggle 403 / Auth Error: {e}")
        print("🔐 Authenticating with Kaggle. Calling kagglehub.login()...")
        kagglehub.login()
        dataset_raw_path = kagglehub.dataset_download("azmainofficial/gastric-cancer")

    if os.path.isdir(os.path.join(dataset_raw_path, 'all_image')):
        DATA_DIR = os.path.join(dataset_raw_path, 'all_image')
    elif os.path.isdir(os.path.join(dataset_raw_path, 'gastric_cancer', 'all_image')):
        DATA_DIR = os.path.join(dataset_raw_path, 'gastric_cancer', 'all_image')
    else:
        DATA_DIR = dataset_raw_path

print(f"✅ Final Dataset Path: {DATA_DIR}")

📥 Checking dataset path / downloading via kagglehub...


100%|██████████| 3.03G/3.03G [02:26<00:00, 22.3MB/s]

Extracting files...


✅ Final Dataset Path: /root/.cache/kagglehub/datasets/azmainofficial/gastric-cancer/versions/1/all_image


In [6]:
# ── Step 5: Class Names & Coarse Group Mapping ─────────────────────────────────
CLASS_NAMES = ['ADI', 'DEB', 'LYM', 'MUC', 'MUS', 'NOR', 'STR', 'TUM']
NUM_CLASSES = len(CLASS_NAMES)

GROUP_NAMES    = ['Epithelial', 'Stromal', 'Other']
EPITHELIAL_IDX = [5, 7]     # NOR, TUM
STROMAL_IDX    = [6, 4, 0]  # STR, MUS, ADI
OTHER_IDX      = [2, 3, 1]  # LYM, MUC, DEB

_class_to_group = [0] * NUM_CLASSES
for g_idx, idx_list in enumerate([EPITHELIAL_IDX, STROMAL_IDX, OTHER_IDX]):
    for c_idx in idx_list:
        _class_to_group[c_idx] = g_idx
CLASS_TO_GROUP = torch.tensor(_class_to_group, dtype=torch.long)

_local_idx = [0] * NUM_CLASSES
for idx_list in [EPITHELIAL_IDX, STROMAL_IDX, OTHER_IDX]:
    for pos, c_idx in enumerate(idx_list):
        _local_idx[c_idx] = pos
LOCAL_IDX_IN_GROUP = torch.tensor(_local_idx, dtype=torch.long)

print("✅ Class & TME Group mappings configured:")
print(f"   • Epithelial : {[CLASS_NAMES[i] for i in EPITHELIAL_IDX]}")
print(f"   • Stromal    : {[CLASS_NAMES[i] for i in STROMAL_IDX]}")
print(f"   • Other      : {[CLASS_NAMES[i] for i in OTHER_IDX]}")

✅ Class & TME Group mappings configured:
   • Epithelial : ['NOR', 'TUM']
   • Stromal    : ['STR', 'MUS', 'ADI']
   • Other      : ['LYM', 'MUC', 'DEB']


In [ ]:
# ── Step 6: Dataset Loader & 80/20 Splitter ─────────────────────────────────────
class TransformedDataset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __getitem__(self, index):
        path, y = self.dataset.samples[self.indices[index]]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, y

    def __len__(self):
        return len(self.indices)

def get_slide_id(filepath):
    fname = os.path.basename(filepath)
    stem, _ = os.path.splitext(fname)
    parts = stem.split('_')
    if len(parts) > 1: return '_'.join(parts[:-1])
    parts = stem.split('-')
    if len(parts) > 2: return '-'.join(parts[:-1])
    return stem

def create_80_20_split(full_dataset, few_shot_fraction=1.0, seed=SEED):
    targets = np.array(full_dataset.targets)
    filepaths = [s[0] for s in full_dataset.samples]
    groups = np.array([get_slide_id(p) for p in filepaths])
    num_classes = len(set(targets.tolist()))
    all_class_set = set(targets.tolist())

    # Sanity check: verify the slide-id heuristic against real filenames before
    # trusting it. If n_unique_groups collapses down near num_classes, get_slide_id()
    # is grouping by CLASS rather than by SLIDE (e.g. filenames like 'ADI_1.png' all
    # map to the same pseudo-slide 'ADI'), which is catastrophic: StratifiedGroupKFold
    # would then have to assign entire classes wholesale to one fold, producing a
    # validation set missing one or more classes entirely.
    n_unique = len(set(groups))
    print(f"🔎 get_slide_id() sanity check: {n_unique} unique groups from {len(groups)} files.")
    print(f"   Sample filename -> inferred slide id:")
    for fp in filepaths[:5]:
        print(f"     {os.path.basename(fp)}  ->  {get_slide_id(fp)}")

    use_grouping = n_unique > num_classes * 2
    if not use_grouping:
        print(f"   ⚠️ WARNING: only {n_unique} unique groups for {num_classes} classes — the "
              f"slide-id heuristic is almost certainly collapsing to class-level grouping on "
              f"this dataset (no real slide metadata is recoverable from these filenames). "
              f"Falling back to a plain stratified split (no grouping). Note this means patch-"
              f"level leakage across train/val cannot be ruled out for this dataset, since no "
              f"slide identifier exists in the filenames to prevent it.")

    if use_grouping:
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        train_idx, val_idx = next(sgkf.split(np.zeros(len(targets)), targets, groups))
        # Even if grouping looked plausible above, verify both splits actually contain
        # every class before trusting them — fall back if not.
        if set(targets[train_idx].tolist()) != all_class_set or set(targets[val_idx].tolist()) != all_class_set:
            print("   ⚠️ Grouped split produced a train or val set missing one or more classes "
                  "— falling back to a plain stratified split instead.")
            use_grouping = False

    if not use_grouping:
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        train_idx, val_idx = next(skf.split(np.zeros(len(targets)), targets))

    # Final hard guarantee: never proceed with a split that's missing a class in
    # either half, regardless of which path produced it.
    assert set(targets[train_idx].tolist()) == all_class_set, "train split is missing at least one class"
    assert set(targets[val_idx].tolist()) == all_class_set, "val split is missing at least one class"

    if few_shot_fraction < 1.0:
        sub_skf = StratifiedKFold(n_splits=int(1.0 / few_shot_fraction), shuffle=True, random_state=seed)
        train_targets = targets[train_idx]
        _, sampled_train_sub_idx = next(sub_skf.split(train_idx, train_targets))
        train_idx = train_idx[sampled_train_sub_idx]
        assert set(train_targets[sampled_train_sub_idx].tolist()) == all_class_set, \
            "few-shot subsample is missing at least one class — try a larger FEW_SHOT_FRACTION"
        print(f"🎯 Few-Shot Sampling ({few_shot_fraction*100:.0f}%): {len(train_idx)} train images")

    return train_idx, val_idx

print("✅ Train/Val Splitter defined.")

In [ ]:
# ── Step 7: Backbone Encoder Factory + HierarchyAwareHead ───────────────────
class HierarchyAwareHead(nn.Module):
    def __init__(self, in_features, num_classes=8, dropout_p=0.3):
        super().__init__()
        self.num_classes = num_classes
        self.dropout = nn.Dropout(p=dropout_p)
        self.coarse_classifier     = nn.Linear(in_features, 3)
        self.epithelial_classifier = nn.Linear(in_features, 2)
        self.stromal_classifier    = nn.Linear(in_features, 3)
        self.other_classifier      = nn.Linear(in_features, 3)

        self.register_buffer('epithelial_idx', torch.tensor(EPITHELIAL_IDX))
        self.register_buffer('stromal_idx',    torch.tensor(STROMAL_IDX))
        self.register_buffer('other_idx',      torch.tensor(OTHER_IDX))

    def forward(self, x):
        x = self.dropout(x)
        coarse_logits  = self.coarse_classifier(x)
        epi_logits     = self.epithelial_classifier(x)
        stromal_logits = self.stromal_classifier(x)
        other_logits   = self.other_classifier(x)

        combined_logits = torch.zeros(x.size(0), self.num_classes, device=x.device)
        combined_logits[:, self.epithelial_idx] = epi_logits + coarse_logits[:, 0:1]
        combined_logits[:, self.stromal_idx]    = stromal_logits + coarse_logits[:, 1:2]
        combined_logits[:, self.other_idx]      = other_logits + coarse_logits[:, 2:3]

        return {
            'combined_logits':   combined_logits,
            'coarse_logits':     coarse_logits,
            'epithelial_logits': epi_logits,
            'stromal_logits':    stromal_logits,
            'other_logits':      other_logits,
        }

class FlatLinearHead(nn.Module):
    def __init__(self, in_features, num_classes=8, dropout_p=0.3):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc      = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return {'combined_logits': self.fc(self.dropout(x))}

def build_paper_encoder(model_type='swin'):
    """
    Builds backbones:
      • 'swin'       : Swin-Base
      • 'convnextv2' : ConvNeXtV2-Tiny (Depthwise Conv + GRN)
      • 'uni'        : UNI Pathology Foundation Model (MahmoodLab, gated HF repo)
      • 'v2_s'       : EfficientNet-V2-S
      • 'effb0'      : EfficientNet-B0 (Lou et al. 2025 baseline backbone)
      • 'vit'        : ViT-B/16 (Lou et al. 2025 baseline backbone)
    """
    if model_type == 'swin':
        encoder = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=0)
        in_features = encoder.num_features
    elif model_type == 'convnextv2':
        encoder = timm.create_model('convnextv2_tiny', pretrained=True, num_classes=0)
        in_features = encoder.num_features
    elif model_type == 'uni':
        try:
            encoder = timm.create_model('hf-hub:MahmoodLab/UNI', pretrained=True, init_values=1e-5, num_classes=0)
        except Exception as e:
            raise RuntimeError(
                "Failed to load UNI from the MahmoodLab HF hub. This repo is gated: "
                "you must (1) request access at https://huggingface.co/MahmoodLab/UNI, "
                "(2) run `from huggingface_hub import login; login()` with a token that "
                "has been granted access, before running this cell again. "
                f"Original error: {e}"
            ) from e
        in_features = encoder.num_features
    elif model_type == 'v2_s':
        # NOTE: 'efficientnetv2_rw_m' is the M variant, not S — use the correct
        # timm id for the true EfficientNetV2-S architecture.
        encoder = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)
        in_features = encoder.num_features
    elif model_type == 'effb0':
        # Matches Lou et al. (2025)'s baseline backbone exactly (pretrained
        # EfficientNet-B0 weights, as they describe using efficientnet_pytorch's
        # pretrained EfficientNet-B0 checkpoint).
        encoder = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        in_features = encoder.num_features
    elif model_type == 'vit':
        encoder = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        in_features = encoder.num_features
    else:
        raise ValueError(f"Unknown model_type '{model_type}'")

    return encoder, in_features

class CustomModel(nn.Module):
    def __init__(self, model_type='swin', num_classes=8, use_hierarchy_head=True, freeze_encoder=True):
        super().__init__()
        self.encoder, in_features = build_paper_encoder(model_type)
        if use_hierarchy_head:
            self.head = HierarchyAwareHead(in_features, num_classes)
        else:
            self.head = FlatLinearHead(in_features, num_classes)

        self.use_hierarchy_head = use_hierarchy_head
        self.freeze_encoder     = freeze_encoder

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
            self.encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_encoder:
            self.encoder.eval()
        return self

    def forward(self, x):
        if self.freeze_encoder:
            with torch.no_grad():
                feats = self.encoder(x)
        else:
            feats = self.encoder(x)

        if feats.dim() > 2:
            feats = torch.flatten(feats, 1)
        return self.head(feats)

print("✅ Architecture Ready (Backbones + HierarchyAwareHead / FlatLinearHead).")


In [9]:
# ── Step 8: Hierarchical Loss & ECE Calculations ───────────────────────────────
_GROUP_PAIR_COST = {
    (0, 1): 3.0, (1, 0): 3.0,   # Epithelial <-> Stromal penalty
    (0, 2): 2.0, (2, 0): 2.0,
    (1, 2): 2.0, (2, 1): 2.0,
}

def _build_cost_matrix():
    mat = torch.zeros(8, 8)
    for i in range(8):
        for j in range(8):
            if i == j: continue
            gi, gj = CLASS_TO_GROUP[i].item(), CLASS_TO_GROUP[j].item()
            mat[i, j] = 1.0 if gi == gj else _GROUP_PAIR_COST[(gi, gj)]
    return mat

COST_MATRIX = _build_cost_matrix()

def hierarchical_loss(outputs, labels, use_hierarchy=True):
    combined_logits = outputs['combined_logits']

    # Primary loss: direct CE on the thing you actually evaluate/checkpoint on
    # (argmax(combined_logits)). Always included so training never fully decouples
    # from the evaluation objective, even when the hierarchical terms below are active.
    primary_loss = F.cross_entropy(combined_logits, labels)

    if not use_hierarchy:
        return primary_loss, primary_loss.item(), 0.0, 0.0, 0.0

    device = labels.device
    coarse_labels = CLASS_TO_GROUP.to(device)[labels]
    coarse_loss   = F.cross_entropy(outputs['coarse_logits'], coarse_labels)

    branch_logits = {0: outputs['epithelial_logits'], 1: outputs['stromal_logits'], 2: outputs['other_logits']}
    fine_loss = torch.zeros((), device=device)
    active_branches = 0
    local_idx = LOCAL_IDX_IN_GROUP.to(device)

    for g in range(3):
        mask = (coarse_labels == g)
        if mask.any():
            fine_loss += F.cross_entropy(branch_logits[g][mask], local_idx[labels[mask]])
            active_branches += 1
    fine_loss /= max(1, active_branches)

    probs        = F.softmax(combined_logits, dim=1)
    hier_penalty = (probs * COST_MATRIX.to(device)[labels]).sum(dim=1).mean()

    total_loss = (PRIMARY_LOSS_WEIGHT * primary_loss
                  + COARSE_LOSS_WEIGHT * coarse_loss
                  + FINE_LOSS_WEIGHT * fine_loss
                  + HIER_LOSS_WEIGHT * hier_penalty)
    return total_loss, primary_loss.item(), coarse_loss.item(), fine_loss.item(), hier_penalty.item()

def calculate_ece(probs, labels, n_bins=10):
    probs, labels = np.array(probs), np.array(labels)
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies  = (predictions == labels)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            ece += np.abs(np.mean(accuracies[in_bin]) - np.mean(confidences[in_bin])) * prop_in_bin
    return ece

def calculate_metrics(probs, labels):
    probs, labels = np.array(probs), np.array(labels)
    preds = np.argmax(probs, axis=1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except ValueError as e:
        print(f"   ⚠️ roc_auc_score failed ({e}); reporting AUC=0.0 for this epoch/batch.")
        auc = 0.0
    ece   = calculate_ece(probs, labels)
    return acc, auc, f1, ece

def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    warmup_steps = max(1, warmup_epochs) * steps_per_epoch
    total_steps  = max(1, total_epochs  * steps_per_epoch)
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + np.cos(np.pi * min(max(progress, 0.0), 1.0)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
# ── Step 9: Training Loop (80/20 Split with Merged Backbone) ────────────────────
def run_training_80_20():
    print(f"\n{'='*75}")
    print(f"🚀 Starting Experiment | Backbone: {MODEL_TYPE.upper()} | Baseline Replication: {BASELINE_REPLICATION_MODE}")
    print(f" Hierarchy Head: {USE_HIERARCHY_HEAD} | Few-Shot Fraction: {FEW_SHOT_FRACTION*100:.0f}%")
    print(f"{'='*75}\n")

    full_dataset = ImageFolder(root=DATA_DIR)
    train_idx, val_idx = create_80_20_split(full_dataset, few_shot_fraction=FEW_SHOT_FRACTION)

    train_data = TransformedDataset(full_dataset, train_idx, transform=get_train_transforms())
    val_data   = TransformedDataset(full_dataset, val_idx,   transform=get_val_transforms())

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

    model = CustomModel(model_type=MODEL_TYPE, num_classes=NUM_CLASSES,
                        use_hierarchy_head=USE_HIERARCHY_HEAD, freeze_encoder=FREEZE_ENCODER).to(DEVICE)

    if BASELINE_REPLICATION_MODE:
        # Matches Lou et al. (2025)'s reported recipe exactly: single Adam
        # optimizer over all parameters at lr=1e-3, weight_decay=1e-4 -- not
        # AdamW, and not a split encoder/head learning rate.
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    elif FREEZE_ENCODER:
        optimizer = optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    else:
        optimizer = optim.AdamW([
            {'params': model.encoder.parameters(), 'lr': ENCODER_LR},
            {'params': model.head.parameters(),    'lr': HEAD_LR},
        ], weight_decay=WEIGHT_DECAY)

    scheduler = get_warmup_cosine_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS, len(train_loader))
    best_score   = -float('inf')
    best_val_loss = float('inf')
    best_metrics  = (0.0, 0.0, 0.0, 0.0)
    patience_cnt  = 0
    save_path     = os.path.join(SAVE_DIR, f"best_{MODEL_TYPE}_hier{USE_HIERARCHY_HEAD}.pth")

    best_val_probs, best_val_labels = [], []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} Train", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss, *_ = hierarchical_loss(outputs, labels, use_hierarchy=USE_HIERARCHY_HEAD)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            t_loss += loss.item() * imgs.size(0)

        t_loss /= len(train_data)

        model.eval()
        v_loss, v_pri_loss, v_probs, v_labels = 0.0, 0.0, [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                total_l, pri_l, *_ = hierarchical_loss(outputs, labels, use_hierarchy=USE_HIERARCHY_HEAD)
                v_loss += total_l.item() * imgs.size(0)
                v_pri_loss += pri_l * imgs.size(0)
                probs = torch.softmax(outputs['combined_logits'], dim=1)
                v_probs.extend(probs.cpu().tolist())
                v_labels.extend(labels.cpu().tolist())

        v_loss /= len(val_data)
        v_pri_loss /= len(val_data)
        acc, auc, f1, ece = calculate_metrics(v_probs, v_labels)

        # Checkpoint criterion according to CHECKPOINT_METRIC
        if CHECKPOINT_METRIC == 'macro_f1':
            current_score = f1
        elif CHECKPOINT_METRIC == 'acc':
            current_score = acc
        elif CHECKPOINT_METRIC == 'primary_ce':
            current_score = -v_pri_loss
        else:
            current_score = -v_loss

        print(f"Ep {epoch:02d} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} (Pri CE: {v_pri_loss:.4f}) | "
              f"Acc: {acc:.4f} | Macro AUC: {auc:.4f} | Macro F1: {f1:.4f} | ECE: {ece:.4f}")

        if current_score > best_score:
            best_score = current_score
            best_val_loss = v_loss
            best_metrics  = (acc, auc, f1, ece)
            best_val_probs, best_val_labels = v_probs, v_labels
            patience_cnt = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': acc, 'val_f1': f1, 'val_loss': v_loss}, save_path)
            score_str = f"{current_score:.4f}" if CHECKPOINT_METRIC in ['macro_f1', 'acc'] else f"{-current_score:.4f}"
            print(f"  ⭐ Saved Checkpoint (Metric [{CHECKPOINT_METRIC}]: {score_str}, Acc: {acc:.4f}, F1: {f1:.4f}, Val Loss: {v_loss:.4f})")
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"🛑 Early Stopping triggered at epoch {epoch}.")
                break

    print(f"\n{'='*75}")
    print(f"🏆 Merged Model Best Results ({MODEL_TYPE.upper()} + HierarchyAwareHead)")
    print(f"   • Accuracy   : {best_metrics[0]:.4f}")
    print(f"   • Macro AUC  : {best_metrics[1]:.4f}")
    print(f"   • Macro F1   : {best_metrics[2]:.4f}")
    print(f"   • ECE        : {best_metrics[3]:.4f}")
    print(f"{'='*75}\n")
    return best_val_probs, best_val_labels

val_probs, val_labels = run_training_80_20()


In [ ]:
# ── Step 10: Diagnostics & Confusion Pair Analysis ────────────────────────────
def analyze_paper_confusions(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    adi_idx, muc_idx = CLASS_NAMES.index('ADI'), CLASS_NAMES.index('MUC')
    mus_idx, str_idx = CLASS_NAMES.index('MUS'), CLASS_NAMES.index('STR')
    nor_idx, tum_idx = CLASS_NAMES.index('NOR'), CLASS_NAMES.index('TUM')

    adi_muc = cm[adi_idx, muc_idx] + cm[muc_idx, adi_idx]
    mus_str = cm[mus_idx, str_idx] + cm[str_idx, mus_idx]
    nor_tum = cm[nor_idx, tum_idx] + cm[tum_idx, nor_idx]
    total_errors = np.sum(cm) - np.trace(cm)

    print(f"\n{'='*70}")
    print(f"📊 TARGETED ERROR PAIR DIAGNOSTICS ({MODEL_TYPE.upper()} + HIERARCHY HEAD)")
    print(f"{'='*70}")
    print(f"Total Errors: {total_errors}")
    print(f"  1. ADI ↔ MUC (Cross-Group Error) : {adi_muc:4d} confusions ({adi_muc/max(1,total_errors)*100:.1f}% of errors)")
    print(f"  2. MUS ↔ STR (Stromal Group)     : {mus_str:4d} confusions ({mus_str/max(1,total_errors)*100:.1f}% of errors)")
    print(f"  3. NOR ↔ TUM (Epithelial Group)  : {nor_tum:4d} confusions ({nor_tum/max(1,total_errors)*100:.1f}% of errors)")
    print(f"{'='*70}\n")

val_preds = np.argmax(val_probs, axis=1)
analyze_paper_confusions(val_labels, val_preds)

print("📋 Full Classification Report:")
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES, digits=4))

# ── Visual Plots ───────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(val_labels, val_preds), annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title(f"Confusion Matrix ({MODEL_TYPE.upper()} + HierHead)", fontweight='bold')

probs_arr = np.array(val_probs)
confidences, accuracies = np.max(probs_arr, axis=1), (val_preds == np.array(val_labels))
bin_b = np.linspace(0, 1, 11)
bin_accs = [np.mean(accuracies[(confidences > bin_b[i]) & (confidences <= bin_b[i+1])]) if np.sum((confidences > bin_b[i]) & (confidences <= bin_b[i+1])) > 0 else 0 for i in range(10)]

ax2.plot([0, 1], [0, 1], 'k--')
ax2.bar(bin_b[:-1], bin_accs, width=0.08, align='edge', alpha=0.6, color='#e74c3c')
ax2.set_title(f"Reliability Diagram (ECE: {calculate_ece(val_probs, val_labels):.4f})", fontweight='bold')
plt.tight_layout()
plt.show()